* 학년: 
* 반:
* 번호:
* 이름:


# 실습 4: 쇼핑몰 고객 군집 모델 구현하기

연 소득과 소비 점수가 비슷한 고객을 k-평균 군집화로 묶고, 실루엣 점수로 군집 품질을 평가해 보자.


**단계별 처리 과정**

| 단계 1 | 단계 2 | 단계 3 | 단계 4 |
|:---:|:---:|:---:|:---:|
| 문제 정의 | 데이터 수집 및 전처리 | 모델 생성 | 모델 평가·활용 |


## 단계 0: 준비 (라이브러리 설치)


In [ ]:
# 라이브러리는 JupyterLite 커널에 미리 포함되어 있습니다.
# (Colab에서는 아래 설치 코드가 실행됩니다.)
import sys

if sys.platform != "emscripten":
    import importlib, subprocess
    for module_name, pip_name in [
        ("pandas", "pandas"),
        ("numpy", "numpy"),
        ("matplotlib", "matplotlib"),
        ("seaborn", "seaborn"),
        ("sklearn", "scikit-learn"),
    ]:
        try:
            importlib.import_module(module_name)
        except ModuleNotFoundError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

from sklearn import set_config
set_config(display="text")

print("라이브러리 준비 완료")


## 단계 1: 문제 정의하기

**쇼핑몰 고객을 특성이 비슷한 집단으로 나누면 어떤 맞춤형 마케팅을 할 수 있을까?**


## 단계 2: 데이터 수집 및 전처리하기

### ① 데이터 불러오기

**💡 복습: CSV 파일**

`pd.read_csv('Mall_Customers.csv')`로 200명의 고객 데이터를 불러오고 `head()`, `info()`, `isnull().sum()`으로 구조와 결측치를 확인한다.


In [ ]:
# pandas를 가져오고 Mall_Customers.csv를 df로 불러오기
import pandas as pd
df = pd.read_csv('Mall_Customers.csv')

# 처음 5행, 데이터 정보, 열별 결측치 수 확인하기
df.head()
df.info()
df.isnull().sum()


**❓ 확인하기**

- 이 데이터에는 고객이 모두 몇 명 있는가?  → **200명**
- 데이터의 속성(열)은 몇 개인가?  → **5개**
- 결측치가 있는 속성이 있는가?  → **없다**


### ② 군집에 사용할 특징 선택하기

**💡 복습: 여러 열 선택**

- `df[['열1', '열2']]`처럼 대괄호를 두 번 사용한다.
- 군집은 정답인 타깃 y 없이 특징 데이터만 사용한다.
- 이번 실습의 특징: 연 소득, 소비 점수


In [ ]:
# 연 소득과 소비 점수 열만 선택해 data에 저장하고 처음 5행 확인하기
data = df[['Annual Income (k$)', 'Spending Score (1-100)']]
print(data.head())


**❓ 확인하기**

- 군집에 사용할 특징(data)은 몇 개인가?  → **2개**
- 군집 분석에는 왜 타깃 y가 필요 없는가?

  → **군집 분석은 정답이 없는 비지도학습이므로, 고객을 비슷한 특성으로 묶는 것이 목표라서 타깃 y가 필요하지 않다.**


## 단계 3: 모델 생성하기

### ① k-평균 군집 모델 학습

**💡 새 API: `KMeans`**

### 1. 모델 만들기

- **API 설명:** `KMeans(n_clusters=k)`는 데이터를 k개 군집으로 나눈다.
- **사용 형식:** `KMeans(n_clusters=5, random_state=42, n_init=10)`

### 2. 모델 학습하기

- **문법 설명:** `model.fit(data)`로 특징 데이터만 넣어 학습한다.


In [ ]:
# KMeans를 가져와 군집 5개인 model 만들기(random_state=42, n_init=10)
from sklearn.cluster import KMeans
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
# data로 모델 학습하기
model.fit(data)


### ② 군집 번호와 중심점 확인

**💡 새 속성**

- `model.labels_`: 학습 데이터 각각의 군집 번호
- `model.cluster_centers_`: 군집별 특징 평균으로 계산한 중심점
- 군집 번호는 크기와 무관하다.


In [ ]:
# model.labels_를 df의 cluster 열에 저장하기
df['cluster'] = model.labels_
# model.cluster_centers_를 centers에 저장하고 출력하기
final_centroid = model.cluster_centers_
print(final_centroid)
# df의 처음 10행에서 특징과 cluster 확인하기
print(df[['Annual Income (k$)', 'Spending Score (1-100)', 'cluster']].head(10))


**❓ 확인하기**

- 군집 0~4에는 각각 몇 명의 고객이 있는가?

  → **군집 0: 81명, 군집 1: 39명, 군집 2: 22명, 군집 3: 35명, 군집 4: 23명**
- 군집 번호가 크다고 더 좋은 군집이라는 뜻인가?  → **아니다. 군집 번호는 단순한 이름표일 뿐이며, 우열은 없다.**


### ③ 군집 결과 시각화

**💡 복습+확장: 산점도**

`sns.scatterplot(..., hue='cluster')`로 군집별 색을 표시하고, `plt.scatter(centers[:,0], centers[:,1], ...)`로 중심점을 표시한다.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 연 소득-소비 점수 산점도를 cluster별 색으로 표시하기
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='cluster', palette='deep')

# centers를 검은 별표로 표시하고 그래프 보여 주기
plt.scatter(final_centroid[:, 0], final_centroid[:, 1], marker='*', color='black', s=200, label='Centroids')
plt.legend()
plt.show()


**❓ 확인하기**

- 산점도에서 검은 별표(중심점)는 무엇을 나타내는가?

  → **각 군집의 평균 연 소득과 평균 소비 점수를 나타내는 중심점이다.**
- 소득도 소비 점수도 모두 높은 군집은 어디에 위치하는가? (오른쪽 위 / 오른쪽 아래 / 왼쪽 위 / 왼쪽 아래)  → **오른쪽 위**


## 단계 4: 모델 평가하기

**💡 새 API: `silhouette_score`**

### 1. 실루엣 점수 구하기

- **API 설명:** `silhouette_score(data, labels)`는 같은 군집끼리 얼마나 가까운지(밀집도)와 다른 군집과 얼마나 멀리 떨어져 있는지(분리도)를 함께 평가해 하나의 점수로 나타낸다.
- **사용 형식:** `silhouette_score(특징 데이터, 군집 번호)`
- **이번 활동:** `silhouette_score(data, model.labels_)`를 계산해 `score`에 저장한다.

### 2. 점수 해석하기

- **문법 설명:** 값의 범위는 -1~1이며, 1에 가까울수록 군집이 잘 분리된 것이고 0에 가까우면 군집 경계가 모호하다는 뜻이다.


In [ ]:
# silhouette_score를 가져와 data와 model.labels_로 실루엣 점수 구하기
from sklearn.metrics import silhouette_score

score = silhouette_score(data, model.labels_)
print(score)


**❓ 확인하기**

- k=5일 때 실루엣 점수는 약 얼마인가?  → **약 0.55**


### 🧪 k를 바꾸어 최적 군집 수 탐색하기

- k=2~8에 대해 모델을 각각 만들고 실루엣 점수를 리스트에 저장한다.
- 가장 높은 점수의 k를 찾는다.
- 반복문의 빈칸을 직접 완성해 보자.


In [ ]:
# ks=range(2,9), scores=[] 준비하기
from sklearn.metrics import silhouette_score

ks = range(2, 9)
scores = []

# 각 k로 KMeans 모델을 학습하고 실루엣 점수를 scores에 추가하기
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(data)
    score = silhouette_score(data, labels)
    scores.append((k, score))

# k별 점수를 출력하고 가장 높은 점수의 k 찾기
print(scores)
print('best k:', max(scores, key=lambda x: x[1])[0])
print('best score:', max(scores, key=lambda x: x[1])[1])


**❓ 확인하기**

- 실루엣 점수가 가장 높은 k는 얼마인가?  → **보통 5**
- 그 점수는 교과서에서 사용한 k=5의 점수와 비교했을 때 어떠한가?

  → **대체로 비슷하거나 약간 높아서, 5개 군집이 적절하다고 볼 수 있다.**


**❓ 확인하기**

- 실루엣 점수가 가장 높은 k는 얼마인가?  → **보통 5**
- 그 점수는 교과서에서 사용한 k=5의 점수와 비교했을 때 어떠한가?

  → **대체로 비슷하거나 약간 높아서, 5개 군집이 적절하다고 볼 수 있다.**
